# HEVC vs VVC Experiment

**Data**

| Codec | QP | Bitrate_kbps | Y_PSNR_dB | Time_sec |
| :--- | :--- | :--- | :--- | :--- |
| HM | 22 | 6311.3760 | 43.2798 | 89.504 |
| HM | 27 | 3410.2560 | 41.5033 | 74.017 |
| HM | 32 | 2052.3360 | 39.1165 | 69.112 |
| HM | 37 | 1222.8000 | 36.2630 | 61.474 |
| VTM | 22 | 4727.0400 | 44.0691 | 570.181 |
| VTM | 27 | 2568.0480 | 42.5558 | 334.987 |
| VTM | 32 | 1578.9600 | 40.5555 | 221.358 |
| VTM | 37 | 972.3840 | 38.0017 | 152.397 |





In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import bjontegaard as bd

In [3]:

# 1. Load the data from the CSV file
df = pd.read_csv('output_metrics.csv')

# 2. Separate the data by codec
hm_data = df[df['Codec'] == 'HM']
vtm_data = df[df['Codec'] == 'VTM']


In [ ]:
# 3. Plot the Rate-Distortion Curve
plt.figure(figsize=(8, 6))

# Plot HM
plt.plot(hm_data['Bitrate_kbps'], hm_data['Y_PSNR_dB'], 
         marker='o', linestyle='-', label='HEVC (HM)')

# Plot VTM
plt.plot(vtm_data['Bitrate_kbps'], vtm_data['Y_PSNR_dB'], 
         marker='s', linestyle='-', label='VVC (VTM)')

plt.xlabel('Bitrate (kbps)')
plt.ylabel('Y-PSNR (dB)')
plt.title('Rate-Distortion Curve: HEVC vs VVC')
plt.legend()
plt.grid(True)

plt.show()

In [9]:
# 4. Calculate BD-Rate
def calculate_bd_rate(rate1, psnr1, rate2, psnr2):
    """
    Calculates the Bjøntegaard Delta Rate (BD-Rate) between two curves.
    rate1/psnr1 = Reference codec (e.g., HM)
    rate2/psnr2 = Target codec (e.g., VTM)
    Returns a negative percentage if the target codec saves bitrate.
    """
    # Convert bitrates to log10 scale
    log_r1 = np.log10(rate1)
    log_r2 = np.log10(rate2)

    # Fit a 3rd-degree (cubic) polynomial: log(Rate) = f(PSNR)
    p1 = np.polyfit(psnr1, log_r1, 3)
    p2 = np.polyfit(psnr2, log_r2, 3)

    # Find the overlapping PSNR integration range
    min_psnr = max(min(psnr1), min(psnr2))
    max_psnr = min(max(psnr1), max(psnr2))
    
    # Helper function to analytically integrate a cubic polynomial
    def integrate_poly(p, x):
        return (p[0]/4)*x**4 + (p[1]/3)*x**3 + (p[2]/2)*x**2 + p[3]*x

    # Calculate the definite integrals for both curves over the overlap
    int1 = integrate_poly(p1, max_psnr) - integrate_poly(p1, min_psnr)
    int2 = integrate_poly(p2, max_psnr) - integrate_poly(p2, min_psnr)

    # Calculate average difference in log space
    avg_diff = (int2 - int1) / (max_psnr - min_psnr)

    # Convert back to a percentage difference
    bd_rate = (10**avg_diff - 1) * 100
    return bd_rate

# Extract arrays from your existing DataFrames
rate_hm = hm_data['Bitrate_kbps'].values
psnr_hm = hm_data['Y_PSNR_dB'].values

rate_vtm = vtm_data['Bitrate_kbps'].values
psnr_vtm = vtm_data['Y_PSNR_dB'].values

# Calculate and print the result
bd_rate_result = calculate_bd_rate(rate_hm, psnr_hm, rate_vtm, psnr_vtm)
print(f"BD-Rate (VVC vs HEVC): {bd_rate_result:.2f}%")

BD-Rate (VVC vs HEVC): -42.91%


In [30]:
# 4. Calculate BD-Rate
rate_hm = hm_data['Bitrate_kbps'].values
psnr_hm = hm_data['Y_PSNR_dB'].values

rate_vtm = vtm_data['Bitrate_kbps'].values
psnr_vtm = vtm_data['Y_PSNR_dB'].values

# Calculate BD-Rate using the library
bd_rate_result = bd.bd_rate(rate_hm, psnr_hm, rate_vtm, psnr_vtm, method='pchip')

# Print the result
print(f"BD-Rate (VVC vs HEVC): {bd_rate_result:.2f}%")

BD-Rate (VVC vs HEVC): -43.36%


In [ ]:
# 5. Complexity Penalty
# Calculate the encoding time ratio for each QP
time_ratio = vtm_data['Time_sec'].values / hm_data['Time_sec'].values

# Create a clean summary dataframe for display
complexity_df = pd.DataFrame({
    'QP': [22, 27, 32, 37],
    'HM Time (s)': hm_data['Time_sec'].values,
    'VTM Time (s)': vtm_data['Time_sec'].values,
    'Ratio': time_ratio
})

# Apply the formatting to the 'Ratio' column to get the "x" string format
complexity_df['Ratio'] = complexity_df['Ratio'].apply(lambda x: f"{x:.2f}x")

# Calculate the average ratio
avg_ratio = time_ratio.mean()

print("--- Encoding Time Complexity ---")
print(complexity_df.to_string(index=False))
print(f"\nAverage Encoding Time Ratio: {avg_ratio:.2f}x")